<a href="https://colab.research.google.com/github/khushi123-git/handwritten-word-recognition-crnn/blob/main/handwriting_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaggle

In [ ]:
import os
os.environ['KAGGLE_USERNAME']="khushigupta1600"
os.environ['KAGGLE_KEY']="KGAT_7cd53845b8a4c7a7b2ec02d81de0a53c"
!kaggle datasets download -d ngkinwang/iam-dataset

In [ ]:
!unzip -q iam-dataset.zip

In [ ]:
!ls

In [ ]:
!ls iam_dataset

In [ ]:
!head iam_dataset/train_gt.txt

In [ ]:
import cv2
import matplotlib.pyplot as plt
img= cv2.imread("iam_dataset/words/a01/a01-000u/a01-000u-00-00.png",0)
plt.imshow(img, cmap="gray")
plt.axis("off")

In [ ]:
labels_new=[]
with open("iam_dataset/train_gt.txt","r") as f:
  for line in f:
    parts=line.strip().split("\t",1)
    if len(parts)==2:
      filename,word=parts
      labels_new.append((filename,word))
print(labels_new[:5])

In [ ]:
with open("iam_dataset/train_gt.txt","r") as f:
  for i in range(10):
    print(repr(f.readline()))

In [ ]:
filename,word=labels_new[0]
path=f"iam_dataset/{filename}"
print(path)
img=cv2.imread(path,0)
plt.imshow(img,cmap="gray")
plt.title("word")
plt.axis("off")

In [ ]:
print(labels_new[0])

In [ ]:
import random
filename,word=random.choice(labels_new)
path=f"iam_dataset/{filename}"
img=cv2.imread(path,0)
img=cv2.resize(img,(128,32))
img=img.astype("float32")/255.0
print(img.shape)
plt.imshow(img,cmap="gray")
plt.title(word)
plt.axis("off")

In [ ]:
chars=set()
for _,word in labels_new:
  chars.update(word)
chars=sorted(list(chars))
print(chars)
print("total: ",len(chars))

In [ ]:
char_to_idx={char:idx+1 for idx,char in enumerate(chars)}
idx_to_char={idx+1:char for idx,char in enumerate(chars)}
print(char_to_idx)

In [ ]:
word=labels_new[0][1]
encoded=[char_to_idx[c] for c in word]
print("word:",word)
print("encoded:",encoded)

In [ ]:
decoded="".join(idx_to_char[i] for i in encoded)
print(decoded)

In [ ]:
import torch
from torch.utils.data import Dataset

In [ ]:
class IAMDataset(Dataset):
  def __init__(self,labels,char_to_idx):
    self.labels=labels
    self.char_to_idx=char_to_idx

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    filename,word=self.labels[idx]
    path=f"iam_dataset/{filename}"
    img=cv2.imread(path,0)
    img=cv2.resize(img,(128,32))
    img=img.astype("float32")/255.0
    img=torch.tensor(img)
    label=[self.char_to_idx[c] for c in word]
    return img,torch.tensor(label)

dataset=IAMDataset(labels_new,char_to_idx)
img,label=dataset[0]
print(label)
print(img.shape)

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
def collate_fn(batch):
  images=[]
  labels=[]
  for img,label in batch:
    images.append(img)
    labels.append(label)
  images=torch.stack(images)
  return images,labels

In [ ]:
dataloader=DataLoader(dataset,batch_size=16,shuffle=True,collate_fn=collate_fn)
for imgs,labels in dataloader:
  print(imgs.shape)
  print(labels)
  break

In [ ]:
import torch.nn as nn

In [ ]:
class CRNN(nn.Module):
  def __init__(self, num_chars) :
    super(CRNN,self).__init__()
    self.cnn=nn.Sequential(
        nn.Conv2d(1,64,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),
        nn.Conv2d(64,128,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
        )
    self.rnn=nn.LSTM(
        input_size=128*8,
        hidden_size=256,
        bidirectional=True
    )
    self.fc=nn.Linear(512,num_chars+1)
  def forward(self,x):
      x=x.unsqueeze(1)
      x=self.cnn(x)
      b,c,h,w=x.size()
      x=x.permute(3,0,1,2)
      x=x.reshape(w,b,c*h)
      x,_=self.rnn(x)
      x=self.fc(x)
      return x

In [ ]:
num_chars=len(chars)
model=CRNN(num_chars)

In [ ]:
imgs,labels=next(iter(dataloader))
output=model(imgs)
print(output.shape)

In [ ]:
import torch.nn.functional as F

In [ ]:
ctc_loss=nn.CTCLoss(blank=0)
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
for imgs,labels in dataloader:
  optimizer.zero_grad()
  output=model(imgs)
  output=F.log_softmax(output,dim=2)
  input_lengths=torch.full(
      size=(imgs.size(0),),
      fill_value=output.size(0),
      dtype=torch.long
  )
  target_lengths=torch.tensor(
      [len(l) for l in labels],
      dtype=torch.long
  )
  targets=torch.cat(labels)
  loss=ctc_loss(output,targets,input_lengths,target_lengths)
  loss.backward()
  optimizer.step()
  print("loss:",loss.item())
  break

In [ ]:
def decode_prediction(output):
  output=output.argmax(2)
  output=output.permute(1,0)
  decoded_words=[]
  for seq in output:
    prev=-1
    word=[]
    for idx in seq:
      idx=idx.item()
      if idx!=prev and idx!=0:
        word.append(idx_to_char.get(idx,""))
      prev=idx
    decoded_words.append("".join(word))
  return decoded_words

In [ ]:
imgs,labels=next(iter(dataloader))
output=model(imgs)
output=F.log_softmax(output,dim=2)
predictions=decode_prediction(output)
print(predictions[:5])

In [ ]:
for i in range(5):
  true_word="".join([idx_to_char[c.item()] for c in labels[i]])
  print("true:",true_word)
  print("pred:",predictions[i])
  print()

In [ ]:
num_epochs=5
for epoch in range(num_epochs):
  total_loss=0
  for i,(imgs,labels) in enumerate(dataloader):
    optimizer.zero_grad()
    output=model(imgs)
    output=F.log_softmax(output,dim=2)
    input_lengths=torch.full(
        size=(imgs.size(0),),
        fill_value=output.size(0),
        dtype=torch.long
    )
    target_lengths=torch.tensor(
        [len(l) for l in labels],
        dtype=torch.long
    )
    targets=torch.cat(labels)
    loss=ctc_loss(output,targets,input_lengths,target_lengths)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
    if(i%50==0):
      print("Batch:",i,"Loss:",loss.item())
  print("Epoch:",epoch+1," Loss:",total_loss/len(dataloader))


In [ ]:
torch.save(model.state_dict(), "handwriting_crnn_epoch5.pth")

In [ ]:
from google.colab import files
files.download("handwriting_crnn_epoch5.pth")

In [ ]:
!pwd
!ls -lh

In [ ]:
from google.colab import files
files.upload()

In [ ]:
model = CRNN(num_chars)
model.load_state_dict(torch.load("handwriting_crnn_epoch5.pth"))
model.eval()

In [ ]:
def predict_word(image_path):
  img=cv2.imread(image_path,0)
  img=cv2.resize(img,(128,32))
  img=img.astype("float32")/255.0
  img=torch.tensor(img).unsqueeze(0)
  with torch.no_grad():
    output=model(img)
    output=F.log_softmax(output,dim=2)
    pred=decode_prediction(output)[0]
  return pred

In [ ]:
for _ in range(10):
  sample= random.choice(labels_new)
  filename=sample[0]
  word=sample[1]
  img_path=f"iam_dataset/{filename}"
  print(img_path)
  pred=predict_word(img_path)
  print("True:",word)
  print("Pred:",pred)
  print()
  #import os
#print(os.path.exists(img_path))

In [ ]:
print(type(labels_new[0]))
print(labels_new[0])

In [ ]:
imgs, labels = next(iter(dataloader))

output = model(imgs)

print(output.shape)

In [ ]:
print(predictions)

In [ ]:
imgs, labels = next(iter(dataloader))

output = model(imgs)
output = F.log_softmax(output, dim=2)

predictions = decode_prediction(output.detach())

for i in range(10):
    true_word = "".join([idx_to_char[c.item()] for c in labels[i]])
    print(true_word, "→", predictions[i])

In [ ]:
correct=0
total=0
for i in range(200):
  imgs,labels=next(iter(dataloader))
  output=model(imgs)
  output=F.log_softmax(output,dim=2)
  preds=decode_prediction(output)
  for j in range(len(preds)):
    true_word="".join([idx_to_char[c.item()] for c in labels[j]])
    if preds[j]==true_word:
      correct=correct+1
    total+=1
accuracy=correct/total
print("Accuracy:",accuracy)

In [ ]:
from difflib import SequenceMatcher
def char_accuracy(true,pred):
  return SequenceMatcher(None,true,pred).ratio()

scores=[]
with torch.no_grad():
  for i in range(200):
    imgs,labels=next(iter(dataloader))
    output=model(imgs)
    output=F.log_softmax(output,dim=2)
    preds=decode_prediction(output)
    for j in range(len(preds)):
      true_word="".join([idx_to_char[c.item()] for c in labels[j]])
      scores.append(char_accuracy(true_word,preds[j]))
print("Character Accuracy:",sum(scores)/len(scores))

In [ ]:
sample=random.choice(labels_new)
filename,word=sample
img_path=f"iam_dataset/{filename}"
pred=predict_word(img_path)
img=cv2.imread(img_path,0)
plt.imshow(img,cmap="gray")
plt.title(f"True:{word}   Predicted:{pred}")
plt.axis("off")

In [ ]:
from google.colab import files
uploaded=files.upload()
for filename in uploaded.keys():
  pred=predict_word(filename)
  img=cv2.imread(filename,0)
  plt.imshow(img, cmap="gray")
  plt.title(f"Prediction: {pred}")
  plt.axis("off")